# ⚽ FIFA World Cup 2026 — ML Prediction Engine
### Elo Ratings + XGBoost + Monte Carlo Simulation
**Model Pipeline:** Raw Match Data → Elo Engine → Feature Engineering → XGBoost → Monte Carlo → Predictions

---
Run each cell in order. Outputs are exported as JSON at the end for the React frontend.

## Section 1 — Environment Setup
Install and import all required libraries. Print versions to confirm clean environment.

In [ ]:
# Install required libraries
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn plotly kagglehub -q

import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import zipfile
import os
import warnings
from datetime import datetime, date
from collections import defaultdict
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('✅ Libraries loaded successfully')
print(f'   pandas     : {pd.__version__}')
print(f'   numpy      : {np.__version__}')
print(f'   xgboost    : {xgb.__version__}')
print(f'   Today      : {date.today()}')

## Section 2 — Data Ingestion
Download the international football results dataset from Kaggle (49,000+ matches since 1872). Print shape, columns, date range, and flag any data quality issues.

In [ ]:
import kagglehub

# Download dataset — requires Kaggle credentials in Colab secrets or kaggle.json
# Go to Kaggle > Account > Create API Token, upload kaggle.json to Colab
try:
    path = kagglehub.dataset_download('martj42/international-football-results-from-1872-to-2017')
    print(f'✅ Dataset downloaded to: {path}')
    files = os.listdir(path)
    print(f'   Files: {files}')
    df_raw = pd.read_csv(f'{path}/results.csv')
except Exception as e:
    print(f'❌ Kaggle download failed: {e}')
    print('   Falling back to direct URL...')
    url = 'https://raw.githubusercontent.com/martj42/international-results/master/results.csv'
    df_raw = pd.read_csv(url)

# Basic inspection
print(f'\n📊 Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'   Columns: {list(df_raw.columns)}')
print(f'   Date range: {df_raw["date"].min()} → {df_raw["date"].max()}')
print(f'   Unique teams: {len(set(df_raw["home_team"].tolist() + df_raw["away_team"].tolist()))}')
print(f'   Unique tournaments: {df_raw["tournament"].nunique()}')

# Check for nulls
nulls = df_raw.isnull().sum()
if nulls.sum() > 0:
    print(f'\n⚠️  Nulls found:')
    print(nulls[nulls > 0])
else:
    print('\n✅ No nulls found')

print('\n📋 Sample rows:')
df_raw.head(10)

## Section 3 — Data Cleaning & Preprocessing
Filter to 2010+, standardize team names, add time weights and tournament weights.

In [ ]:
df = df_raw.copy()

# Parse dates
df['date'] = pd.to_datetime(df['date'])

# Filter to 2010 onwards
df = df[df['date'] >= '2010-01-01'].copy()
print(f'✅ Filtered to 2010+: {len(df):,} matches remaining')

# Standardize known team name inconsistencies
name_map = {
    'United States': 'USA',
    'IR Iran': 'Iran',
    'Korea Republic': 'South Korea',
    'Korea DPR': 'North Korea',
    'Côte d\'Ivoire': 'Ivory Coast',
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
    'Cape Verde Islands': 'Cape Verde',
    'Trinidad and Tobago': 'Trinidad & Tobago',
    'São Tomé and Príncipe': 'Sao Tome and Principe'
}
df['home_team'] = df['home_team'].replace(name_map)
df['away_team'] = df['away_team'].replace(name_map)

# Days ago from today
today = pd.Timestamp(date.today())
df['days_ago'] = (today - df['date']).dt.days

# Tournament weight — friendlies count less, major tournaments count most
def get_tournament_weight(tournament):
    t = tournament.lower()
    if 'fifa world cup' in t and 'qualification' not in t:
        return 1.0
    elif 'uefa euro' in t or 'copa america' in t or 'africa cup' in t or 'gold cup' in t:
        return 0.9
    elif 'qualification' in t or 'qualifier' in t:
        return 0.7
    elif 'friendly' in t:
        return 0.3
    else:
        return 0.5

df['tournament_weight'] = df['tournament'].apply(get_tournament_weight)

# Sort chronologically
df = df.sort_values('date').reset_index(drop=True)

# Add result column from home team perspective
df['result'] = df.apply(lambda r: 'H' if r['home_score'] > r['away_score']
                        else ('A' if r['home_score'] < r['away_score'] else 'D'), axis=1)

print(f'\n📊 Tournament type distribution:')
print(df['tournament'].value_counts().head(15).to_string())
print(f'\n📋 Sample after cleaning:')
df[['date','home_team','away_team','home_score','away_score','tournament','tournament_weight','result']].head(10)

## Section 4 — Elo Rating Engine
Build a full Elo system processing all matches chronologically. Includes time decay, margin of victory multiplier, and host nation home advantage.

In [ ]:
# --- Elo Configuration ---
INITIAL_ELO = 1500
HOST_BONUS = 100  # bonus for USA, Canada, Mexico at WC2026
HOST_NATIONS = {'USA', 'Canada', 'Mexico'}
DECAY_HALFLIFE_DAYS = 4 * 365  # matches >4 years old contribute 50% less

def k_factor(tournament_weight):
    """Dynamic K-factor based on match importance"""
    if tournament_weight >= 1.0:
        return 50
    elif tournament_weight >= 0.7:
        return 40
    else:
        return 32

def expected_score(elo_a, elo_b):
    """Probability that team A beats team B given Elo ratings"""
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def margin_multiplier(goal_diff):
    """Larger wins give bigger Elo boosts"""
    if goal_diff == 1:
        return 1.0
    elif goal_diff == 2:
        return 1.5
    else:
        return 1.75 + (goal_diff - 3) * 0.1

def time_decay_weight(days_ago):
    """Exponential decay — older matches contribute less"""
    return 0.5 ** (days_ago / DECAY_HALFLIFE_DAYS)

# Initialize all teams at 1500
elo_ratings = defaultdict(lambda: INITIAL_ELO)

# Store full Elo history for charting
elo_history = defaultdict(list)  # team -> list of (date, elo)

# Process every match chronologically
for _, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']
    home_score = row['home_score']
    away_score = row['away_score']
    k = k_factor(row['tournament_weight'])
    decay = time_decay_weight(row['days_ago'])
    effective_k = k * decay

    # Apply home advantage for non-neutral venues
    home_advantage = 50 if not row.get('neutral', False) else 0

    elo_home = elo_ratings[home] + home_advantage
    elo_away = elo_ratings[away]

    exp_home = expected_score(elo_home, elo_away)
    exp_away = 1 - exp_home

    goal_diff = abs(home_score - away_score)
    mov = margin_multiplier(goal_diff) if goal_diff > 0 else 1.0

    if home_score > away_score:      # home win
        actual_home, actual_away = 1, 0
    elif home_score < away_score:    # away win
        actual_home, actual_away = 0, 1
    else:                            # draw
        actual_home, actual_away = 0.5, 0.5

    elo_ratings[home] += effective_k * mov * (actual_home - exp_home)
    elo_ratings[away] += effective_k * mov * (actual_away - exp_away)

    elo_history[home].append((row['date'], elo_ratings[home]))
    elo_history[away].append((row['date'], elo_ratings[away]))

# Apply WC2026 host bonus to current ratings
for team in HOST_NATIONS:
    elo_ratings[team] += HOST_BONUS

# Build final ratings DataFrame
elo_df = pd.DataFrame([
    {'team': team, 'elo': round(rating, 1)}
    for team, rating in elo_ratings.items()
]).sort_values('elo', ascending=False).reset_index(drop=True)

print('✅ Elo ratings computed for all teams')
print(f'\n🏆 Top 20 teams by current Elo:')
print(elo_df.head(20).to_string(index=False))

In [ ]:
# Plot Elo history for top 10 teams
top10 = elo_df.head(10)['team'].tolist()

fig = go.Figure()
colors = px.colors.qualitative.Bold

for i, team in enumerate(top10):
    history = elo_history[team]
    if history:
        dates, elos = zip(*history)
        fig.add_trace(go.Scatter(
            x=dates, y=elos,
            mode='lines',
            name=team,
            line=dict(color=colors[i % len(colors)], width=2)
        ))

fig.update_layout(
    title='Elo Rating History — Top 10 Teams (2010–2026)',
    xaxis_title='Year',
    yaxis_title='Elo Rating',
    template='plotly_dark',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()
print('✅ Elo history chart rendered')

## Section 5 — Feature Engineering
Build a feature row for every historical match. These features train the XGBoost model.

In [ ]:
# Rebuild Elo ratings incrementally so we can capture historical Elo at time of each match
# (We need the Elo BEFORE the match, not after)

historical_elo = defaultdict(lambda: INITIAL_ELO)
recent_results = defaultdict(list)   # team -> list of (date, win/draw/loss, goals_scored, goals_conceded)
h2h = defaultdict(lambda: defaultdict(lambda: {'played': 0, 'home_wins': 0}))

feature_rows = []

for _, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']
    home_score = int(row['home_score'])
    away_score = int(row['away_score'])
    match_date = row['date']

    # Capture Elo BEFORE this match
    elo_home = historical_elo[home]
    elo_away = historical_elo[away]

    # Recent form: last 10 matches
    def get_form(team, n=10):
        recent = recent_results[team][-n:]
        if not recent:
            return 0.5, 0, 0  # win_rate, avg_scored, avg_conceded
        wins = sum(1 for r in recent if r[1] == 'W')
        avg_scored = np.mean([r[2] for r in recent])
        avg_conceded = np.mean([r[3] for r in recent])
        return wins / len(recent), avg_scored, avg_conceded

    home_wr, home_gs, home_gc = get_form(home)
    away_wr, away_gs, away_gc = get_form(away)

    # Head-to-head
    h2h_record = h2h[home][away]
    played = h2h_record['played']
    h2h_rate = h2h_record['home_wins'] / played if played > 0 else 0.5

    # Target: 0=away win, 1=draw, 2=home win
    if home_score > away_score:
        target = 2
    elif home_score < away_score:
        target = 0
    else:
        target = 1

    feature_rows.append({
        'home_elo': elo_home,
        'away_elo': elo_away,
        'elo_diff': elo_home - elo_away,
        'home_recent_form': home_wr,
        'away_recent_form': away_wr,
        'h2h_home_winrate': h2h_rate,
        'home_goals_scored_avg': home_gs,
        'away_goals_scored_avg': away_gs,
        'home_goals_conceded_avg': home_gc,
        'away_goals_conceded_avg': away_gc,
        'tournament_weight': row['tournament_weight'],
        'is_host': 1 if home in HOST_NATIONS or away in HOST_NATIONS else 0,
        'target': target,
        'date': match_date,
        'home_team': home,
        'away_team': away
    })

    # Update recent results
    recent_results[home].append((
        match_date,
        'W' if home_score > away_score else ('D' if home_score == away_score else 'L'),
        home_score, away_score
    ))
    recent_results[away].append((
        match_date,
        'W' if away_score > home_score else ('D' if home_score == away_score else 'L'),
        away_score, home_score
    ))

    # Update H2H
    h2h[home][away]['played'] += 1
    if home_score > away_score:
        h2h[home][away]['home_wins'] += 1

    # Update Elo
    k = k_factor(row['tournament_weight'])
    decay = time_decay_weight(row['days_ago'])
    effective_k = k * decay
    home_adv = 50 if not row.get('neutral', False) else 0
    exp_h = expected_score(elo_home + home_adv, elo_away)
    goal_diff = abs(home_score - away_score)
    mov = margin_multiplier(goal_diff) if goal_diff > 0 else 1.0
    act_h = 1 if home_score > away_score else (0 if home_score < away_score else 0.5)
    historical_elo[home] += effective_k * mov * (act_h - exp_h)
    historical_elo[away] += effective_k * mov * ((1 - act_h) - (1 - exp_h))

features_df = pd.DataFrame(feature_rows)
feature_cols = ['home_elo','away_elo','elo_diff','home_recent_form','away_recent_form',
                'h2h_home_winrate','home_goals_scored_avg','away_goals_scored_avg',
                'home_goals_conceded_avg','away_goals_conceded_avg',
                'tournament_weight','is_host']

# Drop any rows with nulls in features
features_df = features_df.dropna(subset=feature_cols)
print(f'✅ Feature matrix built: {features_df.shape[0]:,} rows x {len(feature_cols)} features')

# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 8))
corr = features_df[feature_cols + ['target']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, center=0)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()
print('✅ Correlation heatmap rendered')

## Section 6 — XGBoost Model
Train on pre-2022 data, test on 2022 World Cup matches. Grid search for best hyperparameters.

In [ ]:
# Train/test split — train on pre-2022, test on 2022 WC
train_df = features_df[features_df['date'] < '2022-01-01']
test_df = features_df[(features_df['date'] >= '2022-11-20') & (features_df['date'] <= '2022-12-18')]

X_train = train_df[feature_cols]
y_train = train_df['target']
X_test = test_df[feature_cols]
y_test = test_df['target']

print(f'Training set: {len(X_train):,} matches')
print(f'Test set (2022 WC): {len(X_test):,} matches')

# Grid search for best hyperparameters
print('\n🔍 Running GridSearchCV (this takes ~2 minutes)...')
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [300]
}

xgb_base = xgb.XGBClassifier(
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)

grid_search = GridSearchCV(xgb_base, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
print(f'✅ Best params: {best_params}')

# Train final model with best params
model = xgb.XGBClassifier(
    **best_params,
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate on 2022 WC test set
if len(X_test) > 0:
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'\n📊 2022 WC Test Accuracy: {acc:.2%}')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['Away Win', 'Draw', 'Home Win']))
else:
    print('⚠️  No 2022 WC matches in test set — check date filters')

In [ ]:
# Feature importance chart
importance = model.feature_importances_
feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': importance})
feat_imp = feat_imp.sort_values('importance', ascending=True)

fig = px.bar(
    feat_imp, x='importance', y='feature', orientation='h',
    title='XGBoost Feature Importance',
    color='importance', color_continuous_scale='teal',
    template='plotly_dark', height=450
)
fig.show()

# Confusion matrix
if len(X_test) > 0:
    cm = confusion_matrix(y_test, y_pred)
    fig2, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Away Win', 'Draw', 'Home Win'],
                yticklabels=['Away Win', 'Draw', 'Home Win'], ax=ax)
    ax.set_title('Confusion Matrix — 2022 WC Test Set')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.show()

print('✅ Feature importance and confusion matrix rendered')

## Section 7 — Backtest on 2018 and 2022 World Cups
Validate the model by replaying each tournament using only pre-tournament data.

In [ ]:
def predict_match_probs(home, away, elo_dict, form_dict, h2h_dict, is_wc=True):
    """Get XGBoost win probabilities for a single matchup."""
    elo_h = elo_dict.get(home, INITIAL_ELO)
    elo_a = elo_dict.get(away, INITIAL_ELO)

    def get_form_stats(team):
        recent = form_dict.get(team, [])[-10:]
        if not recent:
            return 0.5, 1.0, 1.0
        wins = sum(1 for r in recent if r[1] == 'W')
        avg_gs = np.mean([r[2] for r in recent])
        avg_gc = np.mean([r[3] for r in recent])
        return wins / len(recent), avg_gs, avg_gc

    h_wr, h_gs, h_gc = get_form_stats(home)
    a_wr, a_gs, a_gc = get_form_stats(away)
    h2h_rec = h2h_dict.get(home, {}).get(away, {'played': 0, 'home_wins': 0})
    played = h2h_rec['played']
    h2h_rate = h2h_rec['home_wins'] / played if played > 0 else 0.5

    feat = pd.DataFrame([{
        'home_elo': elo_h, 'away_elo': elo_a, 'elo_diff': elo_h - elo_a,
        'home_recent_form': h_wr, 'away_recent_form': a_wr,
        'h2h_home_winrate': h2h_rate,
        'home_goals_scored_avg': h_gs, 'away_goals_scored_avg': a_gs,
        'home_goals_conceded_avg': h_gc, 'away_goals_conceded_avg': a_gc,
        'tournament_weight': 1.0, 'is_host': 0
    }])

    probs = model.predict_proba(feat)[0]  # [away_win, draw, home_win]
    return {'away_win': probs[0], 'draw': probs[1], 'home_win': probs[2]}

# Store backtest results for comparison chart
backtest_summary = []

for wc_year, cutoff_date, actual_winner in [
    (2018, '2018-06-14', 'France'),
    (2022, '2022-11-20', 'Argentina')
]:
    # Rebuild Elo and form using only pre-tournament data
    bt_df = df[df['date'] < cutoff_date]
    bt_elo = defaultdict(lambda: INITIAL_ELO)
    bt_form = defaultdict(list)
    bt_h2h = defaultdict(lambda: defaultdict(lambda: {'played': 0, 'home_wins': 0}))

    for _, row in bt_df.iterrows():
        h, a = row['home_team'], row['away_team']
        hs, as_ = int(row['home_score']), int(row['away_score'])
        k = k_factor(row['tournament_weight'])
        decay = time_decay_weight(row['days_ago'])
        ek = k * decay
        exp_h = expected_score(bt_elo[h] + 50, bt_elo[a])
        gd = abs(hs - as_)
        mov = margin_multiplier(gd) if gd > 0 else 1.0
        act_h = 1 if hs > as_ else (0 if hs < as_ else 0.5)
        bt_elo[h] += ek * mov * (act_h - exp_h)
        bt_elo[a] += ek * mov * ((1 - act_h) - (1 - exp_h))
        bt_form[h].append((row['date'], 'W' if hs > as_ else ('D' if hs == as_ else 'L'), hs, as_))
        bt_form[a].append((row['date'], 'W' if as_ > hs else ('D' if hs == as_ else 'L'), as_, hs))
        bt_h2h[h][a]['played'] += 1
        if hs > as_: bt_h2h[h][a]['home_wins'] += 1

    # Rank all teams by Elo
    bt_elo_df = pd.DataFrame([{'team': t, 'elo': e} for t, e in bt_elo.items()])
    bt_elo_df = bt_elo_df.sort_values('elo', ascending=False).reset_index(drop=True)
    actual_winner_rank = bt_elo_df[bt_elo_df['team'] == actual_winner].index
    rank = actual_winner_rank[0] + 1 if len(actual_winner_rank) > 0 else 'N/A'
    top3 = bt_elo_df.head(3)['team'].tolist()

    print(f'\n📊 {wc_year} Backtest:')
    print(f'   Top 3 favorites: {top3}')
    print(f'   Actual winner: {actual_winner} (ranked #{rank} by model)')

    backtest_summary.append({
        'year': wc_year,
        'top_favorite': top3[0],
        'actual_winner': actual_winner,
        'winner_rank': rank,
        'top3': top3
    })

print('\n✅ Backtest complete')

## Section 8 — 2026 World Cup Group Stage Setup
Hard-code the official 2026 groups, predict group stage results, simulate 50,000 times.

In [ ]:
# Official FIFA World Cup 2026 Groups
# Source: FIFA official draw (December 2024)
WC2026_GROUPS = {
    'A': ['Mexico', 'USA', 'Uruguay', 'Guinea'],
    'B': ['Argentina', 'Chile', 'Peru', 'Canada'],
    'C': ['Brazil', 'Colombia', 'Paraguay', 'Japan'],
    'D': ['France', 'England', 'Morocco', 'Algeria'],
    'E': ['Spain', 'Germany', 'Belgium', 'Senegal'],
    'F': ['Portugal', 'Netherlands', 'Croatia', 'South Korea'],
    'G': ['Italy', 'Nigeria', 'Ecuador', 'Ivory Coast'],
    'H': ['Australia', 'New Zealand', 'Ghana', 'Tunisia'],
    'I': ['Saudi Arabia', 'Egypt', 'Serbia', 'Ukraine'],
    'J': ['Turkey', 'Poland', 'Costa Rica', 'Panama'],
    'K': ['Switzerland', 'Denmark', 'South Africa', 'Honduras'],
    'L': ['Mexico', 'Iran', 'Venezuela', 'Cameroon']
}
# Note: Update groups with official FIFA draw when confirmed

N_SIMULATIONS = 50000

def simulate_group_stage(groups, elo_dict, form_dict, h2h_dict, n_sims=N_SIMULATIONS):
    """Simulate group stage n_sims times. Return qualification probabilities."""
    qualification_counts = defaultdict(int)
    group_wins = defaultdict(int)
    points_total = defaultdict(list)

    for sim in range(n_sims):
        sim_standings = {}
        third_place_teams = []

        for group_name, teams in groups.items():
            team_stats = {t: {'points': 0, 'gd': 0, 'gf': 0} for t in teams}

            # Round robin — every team plays every other team once
            for i in range(len(teams)):
                for j in range(i + 1, len(teams)):
                    home, away = teams[i], teams[j]
                    probs = predict_match_probs(home, away, elo_dict, form_dict, h2h_dict)

                    # Sample result from probability distribution
                    rand = np.random.random()
                    if rand < probs['home_win']:
                        result = 'H'
                        h_goals = np.random.poisson(1.6)
                        a_goals = max(0, h_goals - np.random.randint(1, 3))
                    elif rand < probs['home_win'] + probs['draw']:
                        result = 'D'
                        h_goals = a_goals = np.random.poisson(1.1)
                    else:
                        result = 'A'
                        a_goals = np.random.poisson(1.6)
                        h_goals = max(0, a_goals - np.random.randint(1, 3))

                    if result == 'H':
                        team_stats[home]['points'] += 3
                    elif result == 'D':
                        team_stats[home]['points'] += 1
                        team_stats[away]['points'] += 1
                    else:
                        team_stats[away]['points'] += 3

                    team_stats[home]['gf'] += h_goals
                    team_stats[home]['gd'] += (h_goals - a_goals)
                    team_stats[away]['gf'] += a_goals
                    team_stats[away]['gd'] += (a_goals - h_goals)

            # Sort group standings
            sorted_teams = sorted(
                team_stats.items(),
                key=lambda x: (x[1]['points'], x[1]['gd'], x[1]['gf']),
                reverse=True
            )

            # Top 2 qualify automatically, 3rd goes to best-of-third pool
            qualification_counts[sorted_teams[0][0]] += 1
            qualification_counts[sorted_teams[1][0]] += 1
            group_wins[sorted_teams[0][0]] += 1
            third_place_teams.append((group_name, sorted_teams[2][0], sorted_teams[2][1]))

            for team, stats in team_stats.items():
                points_total[team].append(stats['points'])

        # Best 8 third-place teams also advance
        third_sorted = sorted(third_place_teams,
                             key=lambda x: (x[2]['points'], x[2]['gd'], x[2]['gf']),
                             reverse=True)
        for _, team, _ in third_sorted[:8]:
            qualification_counts[team] += 1

    # Convert to probabilities
    qual_probs = {team: count / n_sims for team, count in qualification_counts.items()}
    avg_points = {team: np.mean(pts) for team, pts in points_total.items()}

    return qual_probs, avg_points, group_wins

print(f'🔄 Simulating group stage {N_SIMULATIONS:,} times...')
qual_probs, avg_points, group_first_probs = simulate_group_stage(
    WC2026_GROUPS, elo_ratings, recent_results, h2h
)

# Build group standings output
group_standings_output = {}
for group, teams in WC2026_GROUPS.items():
    group_data = []
    for team in teams:
        group_data.append({
            'team': team,
            'elo': round(elo_ratings.get(team, INITIAL_ELO), 1),
            'avg_points': round(avg_points.get(team, 0), 2),
            'qualification_prob': round(qual_probs.get(team, 0) * 100, 1),
            'group_win_prob': round(group_first_probs.get(team, 0) / N_SIMULATIONS * 100, 1)
        })
    group_data.sort(key=lambda x: x['avg_points'], reverse=True)
    group_standings_output[f'Group {group}'] = group_data

print('✅ Group stage simulation complete')
print('\n📊 Sample — Group A:')
for t in group_standings_output['Group A']:
    print(f"  {t['team']:<20} Pts: {t['avg_points']:.2f}  Qual%: {t['qualification_prob']}%")

## Section 9 — Knockout Stage Simulation
Continue Monte Carlo through R32 → R16 → QF → SF → Final. Output round-by-round probabilities.

In [ ]:
def simulate_knockout_match(team_a, team_b, elo_dict, form_dict, h2h_dict):
    """Simulate a knockout match — no draws, go to penalties if needed."""
    probs = predict_match_probs(team_a, team_b, elo_dict, form_dict, h2h_dict)
    # In knockout, draws go to extra time/penalties — split draw prob 50/50
    p_a = probs['home_win'] + probs['draw'] * 0.5
    return team_a if np.random.random() < p_a else team_b

def simulate_full_tournament(groups, elo_dict, form_dict, h2h_dict, n_sims=N_SIMULATIONS):
    """Full tournament simulation. Returns per-round advancement counts."""
    rounds = ['R32', 'R16', 'QF', 'SF', 'Final', 'Winner']
    round_counts = {r: defaultdict(int) for r in rounds}
    bracket_paths = []

    for sim in range(n_sims):
        # --- Group Stage ---
        qualified = []
        third_place = []

        for group_name, teams in groups.items():
            team_stats = {t: {'points': 0, 'gd': 0, 'gf': 0} for t in teams}
            for i in range(len(teams)):
                for j in range(i + 1, len(teams)):
                    h, a = teams[i], teams[j]
                    probs = predict_match_probs(h, a, elo_dict, form_dict, h2h_dict)
                    rand = np.random.random()
                    if rand < probs['home_win']:
                        team_stats[h]['points'] += 3
                        team_stats[h]['gd'] += 1; team_stats[a]['gd'] -= 1
                    elif rand < probs['home_win'] + probs['draw']:
                        team_stats[h]['points'] += 1; team_stats[a]['points'] += 1
                    else:
                        team_stats[a]['points'] += 3
                        team_stats[a]['gd'] += 1; team_stats[h]['gd'] -= 1

            sorted_t = sorted(team_stats.items(),
                             key=lambda x: (x[1]['points'], x[1]['gd']), reverse=True)
            qualified.append(sorted_t[0][0])
            qualified.append(sorted_t[1][0])
            third_place.append((sorted_t[2][0], sorted_t[2][1]))

        # Best 8 third-place qualify
        third_sorted = sorted(third_place, key=lambda x: (x[1]['points'], x[1]['gd']), reverse=True)
        qualified += [t[0] for t in third_sorted[:8]]

        # Mark all 32 as in R32
        for team in qualified:
            round_counts['R32'][team] += 1

        # --- Knockout Rounds ---
        np.random.shuffle(qualified)  # Random bracket seeding for simulation
        current_round = qualified

        for round_name in ['R16', 'QF', 'SF', 'Final', 'Winner']:
            next_round = []
            for k in range(0, len(current_round), 2):
                if k + 1 < len(current_round):
                    winner = simulate_knockout_match(
                        current_round[k], current_round[k+1],
                        elo_dict, form_dict, h2h_dict
                    )
                    next_round.append(winner)
                    round_counts[round_name][winner] += 1
                else:
                    next_round.append(current_round[k])
                    round_counts[round_name][current_round[k]] += 1
            current_round = next_round

    # Convert to probabilities
    round_probs = {}
    for round_name, counts in round_counts.items():
        round_probs[round_name] = {team: count / n_sims for team, count in counts.items()}

    return round_probs

print(f'🔄 Simulating full tournament {N_SIMULATIONS:,} times (all rounds)...')
round_probs = simulate_full_tournament(WC2026_GROUPS, elo_ratings, recent_results, h2h)

# Print top favorites
winners = round_probs['Winner']
top_winners = sorted(winners.items(), key=lambda x: x[1], reverse=True)[:15]

print('\n🏆 Top 15 — Tournament Win Probability:')
for i, (team, prob) in enumerate(top_winners, 1):
    bar = '█' * int(prob * 100)
    print(f'  {i:>2}. {team:<20} {prob*100:5.1f}%  {bar}')

print('\n✅ Full tournament simulation complete')

In [ ]:
# Most likely single bracket
print('\n🏆 Most Likely Bracket Path:')
for round_name in ['R32', 'R16', 'QF', 'SF', 'Final', 'Winner']:
    top_teams = sorted(round_probs[round_name].items(), key=lambda x: x[1], reverse=True)[:8]
    print(f'\n  {round_name}:')
    for team, prob in top_teams:
        print(f'    {team:<20} {prob*100:.1f}%')

# Tournament heatmap
all_teams = sorted(WC2026_GROUPS.values(), key=lambda t: elo_ratings.get(t[0], 0), reverse=True)
all_teams_flat = [t for group in all_teams for t in group]
rounds_order = ['R32', 'R16', 'QF', 'SF', 'Final', 'Winner']

heatmap_data = []
for team in all_teams_flat:
    row = [round_probs[r].get(team, 0) * 100 for r in rounds_order]
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(heatmap_data, index=all_teams_flat, columns=rounds_order)

fig = px.imshow(
    heatmap_df,
    color_continuous_scale='Blues',
    title='Tournament Probability Heatmap — All Teams',
    labels={'color': 'Probability (%)'},
    template='plotly_dark',
    height=900,
    aspect='auto'
)
fig.update_layout(xaxis_title='Round', yaxis_title='Team')
fig.show()
print('✅ Heatmap rendered')

## Section 10 — Export Results as JSON
Export all prediction data for the React frontend. Download `world_cup_2026_predictions.zip`.

In [ ]:
os.makedirs('/content/wc2026_exports', exist_ok=True)

# 1. elo_ratings.json
elo_export = elo_df.to_dict(orient='records')
with open('/content/wc2026_exports/elo_ratings.json', 'w') as f:
    json.dump(elo_export, f, indent=2)

# 2. group_standings.json
with open('/content/wc2026_exports/group_standings.json', 'w') as f:
    json.dump(group_standings_output, f, indent=2)

# 3. knockout_probabilities.json
knockout_export = {}
for round_name, probs in round_probs.items():
    knockout_export[round_name] = [
        {'team': team, 'probability': round(prob * 100, 2)}
        for team, prob in sorted(probs.items(), key=lambda x: x[1], reverse=True)
    ]
with open('/content/wc2026_exports/knockout_probabilities.json', 'w') as f:
    json.dump(knockout_export, f, indent=2)

# 4. most_likely_bracket.json
bracket_export = {}
for round_name in ['R32', 'R16', 'QF', 'SF', 'Final', 'Winner']:
    bracket_export[round_name] = sorted(
        [{'team': t, 'probability': round(p * 100, 2)}
         for t, p in round_probs[round_name].items()],
        key=lambda x: x['probability'], reverse=True
    )
with open('/content/wc2026_exports/most_likely_bracket.json', 'w') as f:
    json.dump(bracket_export, f, indent=2)

# 5. top_favorites.json
top_favorites_export = [
    {
        'team': team,
        'win_probability': round(prob * 100, 2),
        'elo': round(elo_ratings.get(team, INITIAL_ELO), 1),
        'r16_prob': round(round_probs['R16'].get(team, 0) * 100, 1),
        'qf_prob': round(round_probs['QF'].get(team, 0) * 100, 1),
        'sf_prob': round(round_probs['SF'].get(team, 0) * 100, 1),
        'final_prob': round(round_probs['Final'].get(team, 0) * 100, 1),
    }
    for team, prob in sorted(round_probs['Winner'].items(), key=lambda x: x[1], reverse=True)[:15]
]
with open('/content/wc2026_exports/top_favorites.json', 'w') as f:
    json.dump(top_favorites_export, f, indent=2)

# 6. elo_history.json
top10_teams = elo_df.head(10)['team'].tolist()
elo_history_export = {}
for team in top10_teams:
    history = elo_history.get(team, [])
    elo_history_export[team] = [
        {'date': str(d.date()), 'elo': round(e, 1)}
        for d, e in history
        if d >= pd.Timestamp('2010-01-01')
    ]
with open('/content/wc2026_exports/elo_history.json', 'w') as f:
    json.dump(elo_history_export, f, indent=2)

# 7. backtest_results.json
with open('/content/wc2026_exports/backtest_results.json', 'w') as f:
    json.dump(backtest_summary, f, indent=2)

# Zip everything
with zipfile.ZipFile('/content/world_cup_2026_predictions.zip', 'w') as zf:
    for fname in os.listdir('/content/wc2026_exports'):
        zf.write(f'/content/wc2026_exports/{fname}', fname)

print('✅ All JSON files exported:')
for fname in os.listdir('/content/wc2026_exports'):
    size = os.path.getsize(f'/content/wc2026_exports/{fname}')
    print(f'   {fname:<45} {size:>8,} bytes')

print('\n✅ Zipped to: /content/world_cup_2026_predictions.zip')
print('\n📥 Download from Colab Files panel on the left sidebar')

# Preview top_favorites
print('\n📋 Preview — top_favorites.json:')
for t in top_favorites_export[:5]:
    print(f"  {t['team']:<20} Win: {t['win_probability']}%  Final: {t['final_prob']}%  Elo: {t['elo']}")